# 06 — Yoko Musanga × Google Satellite Embedding regression

## Question
Can the **mean UAV-derived Musanga probability at 10 m** be predicted from the 64-dimensional Google Satellite Embedding for the Yoko study site?

This is deliberately a **single-site proof of concept**. It uses a random train/test split for rapid model development. Cross-site validation should be added later when the other UAV Musanga probability products are introduced.

## Workflow
1. Load the validated UAV `musanga_probability_uint16.tif`.
2. Download the **2020 Google Satellite Embedding V1** over the UAV footprint.
3. Use the embedding raster as the authoritative satellite grid.
4. Aggregate UAV probability onto that grid using **average resampling** — no Musanga threshold.
5. Build a table containing `musanga_prob_mean` and A00…A63.
6. Benchmark Random Forest and HistGradientBoosting regressors.
7. Export observed, predicted, and residual 10-m GeoTIFFs.

The target remains continuous throughout.

## BLOCK 01 — Install and import packages

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 01 — INSTALL AND IMPORT PACKAGES")
print("=" * 100)

!pip -q install earthengine-api geemap rasterio pyproj scikit-learn scipy joblib

from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from scipy.stats import spearmanr
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import ee
import geemap

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
print("Imports OK")

## BLOCK 02 — Mount Google Drive and define paths

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 02 — MOUNT DRIVE AND DEFINE PATHS")
print("=" * 100)

from google.colab import drive
DRIVE_ROOT = Path("/content/drive")
MYDRIVE = DRIVE_ROOT / "MyDrive"
if not MYDRIVE.exists() or not any(MYDRIVE.iterdir()):
    drive.mount(str(DRIVE_ROOT), force_remount=False)
else:
    print("Google Drive already mounted.")

PROJECT_ROOT = MYDRIVE / "Colab Notebooks" / "Musanga"
WORK_DIR = PROJECT_ROOT / "work"
INPUT_PROB = WORK_DIR / "whole_orthomosaic_inference" / "musanga_probability_uint16.tif"

OUT_DIR = WORK_DIR / "yoko_satellite_embedding"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EMBED_TIF = OUT_DIR / "Yoko_GoogleSatelliteEmbedding_2020_10m.tif"
OBS_TIF = OUT_DIR / "Yoko_UAV_Musanga_mean_10m.tif"
PRED_TIF = OUT_DIR / "Yoko_Embedding_Musanga_predicted_10m.tif"
RESID_TIF = OUT_DIR / "Yoko_Embedding_Musanga_residual_10m.tif"
TABLE_CSV = OUT_DIR / "Yoko_embedding_training_table.csv"
MODEL_FILE = OUT_DIR / "Yoko_best_embedding_regressor.joblib"
SUMMARY_JSON = OUT_DIR / "Yoko_embedding_regression_summary.json"

assert INPUT_PROB.exists(), f"Missing input: {INPUT_PROB}"
print("INPUT:", INPUT_PROB)
print("OUTPUT:", OUT_DIR)

## BLOCK 03 — Configuration and inspect UAV probability raster

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 03 — CONFIGURATION / UAV INPUT")
print("=" * 100)

EE_PROJECT = "tropsedslu"
EMBEDDING_YEAR = 2020
TEST_SIZE = 0.20
FORCE_DOWNLOAD_EMBEDDINGS = False

with rasterio.open(INPUT_PROB) as src:
    print("CRS:", src.crs)
    print("Shape:", src.height, src.width)
    print("Resolution:", src.res)
    print("dtype:", src.dtypes[0], "nodata:", src.nodata)
    print("bounds:", src.bounds)

    # Sparse sample only: do not load the full UAV raster.
    oh = min(1000, src.height)
    ow = min(1000, src.width)
    sample = src.read(1, out_shape=(oh, ow), resampling=Resampling.nearest).astype("float32")
    if src.nodata is not None:
        sample = sample[sample != src.nodata]
    sample = sample[np.isfinite(sample)]

print("Sample min/max:", float(sample.min()), float(sample.max()))
# Authoritative encoding of musanga_probability_uint16.tif:
# stored uint16 value * 0.0001 = probability.
PROB_SCALE = 10000.0
print("Probability divisor:", PROB_SCALE)
print("Embedding year:", EMBEDDING_YEAR)

## BLOCK 04 — Authenticate Earth Engine and download the 2020 embedding

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 04 — EARTH ENGINE / GOOGLE SATELLITE EMBEDDING")
print("=" * 100)

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

with rasterio.open(INPUT_PROB) as src:
    b = src.bounds
    left, bottom, right, top = transform_bounds(
        src.crs, "EPSG:4326", b.left, b.bottom, b.right, b.top, densify_pts=21
    )

roi = ee.Geometry.Rectangle([left, bottom, right, top], geodesic=False)

collection = (
    ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
    .filterDate(f"{EMBEDDING_YEAR}-01-01", f"{EMBEDDING_YEAR + 1}-01-01")
    .filterBounds(roi)
)
print("Embedding images intersecting Yoko:", collection.size().getInfo())

band_names = [f"A{i:02d}" for i in range(64)]
embedding = collection.mosaic().select(band_names)

# Backward compatibility: move/copy an already-downloaded embedding from the old folder.
OLD_EMBED_TIF = OUT_DIR / "Yoko_GoogleSatelliteEmbedding_2020_10m.tif"
if (not EMBED_TIF.exists()) and OLD_EMBED_TIF.exists():
    import shutil
    shutil.copy2(OLD_EMBED_TIF, EMBED_TIF)
    print("Copied existing embedding into persistent sat cache:", EMBED_TIF)

if FORCE_DOWNLOAD_EMBEDDINGS or not EMBED_TIF.exists():
    print("Downloading 64-band embedding raster...")
    geemap.ee_export_image(
        embedding,
        filename=str(EMBED_TIF),
        scale=10,
        region=roi,
        file_per_band=False,
    )
else:
    print("Reusing:", EMBED_TIF)

assert EMBED_TIF.exists(), "Embedding export failed."
with rasterio.open(EMBED_TIF) as src:
    print("Embedding CRS:", src.crs)
    print("Embedding shape:", src.height, src.width)
    print("Embedding bands:", src.count)
    print("Embedding resolution:", src.res)
    assert src.count == 64, f"Expected 64 bands, got {src.count}"

## BLOCK 05 — Aggregate UAV probability directly to the embedding grid

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 05 — UAV MEAN PROBABILITY ON EMBEDDING GRID (PERSISTENT CACHE)")
print("=" * 100)

with rasterio.open(EMBED_TIF) as emb:
    dst_crs = emb.crs
    dst_transform = emb.transform
    dst_shape = (emb.height, emb.width)
    out_profile = emb.profile.copy()

# Reuse the expensive aggregation whenever it is already on Drive.
if OBS_TIF.exists():
    print("Reusing cached aggregation:", OBS_TIF)
    with rasterio.open(OBS_TIF) as src:
        cached = src.read(1).astype("float32")
        cached_nodata = src.nodata
    good = np.isfinite(cached)
    if cached_nodata is not None:
        good &= cached != cached_nodata
    obs = np.where(good, cached, np.nan).astype("float32")
    source_nodata = 65535
else:
    print("No cached aggregation found — computing once and writing to:", OBS_TIF)
    with rasterio.open(INPUT_PROB) as src:
        source_nodata = src.nodata
        if source_nodata is None:
            raise ValueError("UAV probability raster has no declared NoData value; cannot safely aggregate.")
        obs_raw = np.full(dst_shape, np.nan, dtype="float32")
        reproject(
            source=rasterio.band(src, 1),
            destination=obs_raw,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=source_nodata,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            dst_nodata=np.nan,
            resampling=Resampling.average,
            init_dest_nodata=True,
        )

    # Authoritative uint16 encoding: stored value * 0.0001 = probability.
    obs = obs_raw / PROB_SCALE
    good = np.isfinite(obs) & (obs >= 0.0) & (obs <= 1.0)

    profile1 = out_profile.copy()
    profile1.update(count=1, dtype="float32", nodata=-9999.0, compress="deflate")
    with rasterio.open(OBS_TIF, "w", **profile1) as dst:
        dst.write(np.where(good, obs, -9999.0).astype("float32"), 1)
    print("Cached aggregation saved.")

# Define export profile also when loading from cache.
profile1 = out_profile.copy()
profile1.update(count=1, dtype="float32", nodata=-9999.0, compress="deflate")

print("Source UAV NoData value:", source_nodata)
print("Satellite-grid cells:", obs.size)
print("Valid UAV target cells:", int(good.sum()))
print("Excluded / NoData cells:", int((~good).sum()))
print(f"Valid fraction of embedding grid: {good.mean():.3%}")
print("Mean target:", float(np.nanmean(obs[good])))
print("Median target:", float(np.nanmedian(obs[good])))
print("P90 target:", float(np.nanpercentile(obs[good], 90)))
print("Max target:", float(np.nanmax(obs[good])))
print("Persistent target:", OBS_TIF)

## BLOCK 06 — Inspect target distribution

## BLOCK 05B — Visual QA: raw UAV, 10 m aggregation, satellite imagery and Earth Engine map\n\nThis is a visual check only. It does not alter the modelling target. The satellite panel uses a cloud-filtered Sentinel-2 RGB composite close to the UAV survey (24–25 August 2020).\n

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 05B — VISUAL QA / UAV → 10 m → SATELLITE")
print("=" * 100)

# 1) Raw UAV probability, downsampled only for display (not for modelling).
with rasterio.open(INPUT_PROB) as src:
    dh = min(1800, src.height)
    dw = max(1, int(src.width * dh / src.height))
    raw = src.read(1, out_shape=(dh, dw), resampling=Resampling.average).astype("float32")
    raw_nodata = src.nodata
    raw = raw / PROB_SCALE
    if raw_nodata is not None:
        # The display resampling can mix edge NoData; use source mask for display validity.
        m = src.dataset_mask(out_shape=(dh, dw), resampling=Resampling.nearest) > 0
        raw[~m] = np.nan
    raw[(raw < 0) | (raw > 1)] = np.nan

# 2) Cached 10-m target.
agg_show = np.where(good, obs, np.nan)

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(raw, vmin=0, vmax=1)
ax.set_title("Raw UAV Musanga probability — display overview")
ax.axis("off")
plt.colorbar(im, ax=ax, label="Musanga probability")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(agg_show, vmin=0, vmax=1)
ax.set_title("UAV Musanga probability aggregated to Google embedding grid (~10 m)")
ax.axis("off")
plt.colorbar(im, ax=ax, label="Mean UAV Musanga probability")
plt.tight_layout()
plt.show()

# 3) Sentinel-2 RGB around the UAV survey dates (24–25 Aug 2020).
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate("2020-08-01", "2020-09-16")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
)
print("Sentinel-2 candidate scenes (2020-08-01 to 2020-09-15):", s2.size().getInfo())
s2_rgb = s2.median().select(["B4", "B3", "B2"])

# Interactive Earth Engine map: Sentinel-2 plus semi-transparent aggregated UAV target.
m = geemap.Map()
m.centerObject(roi, 15)
m.addLayer(s2_rgb, {"min": 0, "max": 3000, "gamma": 1.2}, "Sentinel-2 RGB Aug/Sep 2020")

# Uploading the local 10-m raster back to EE is unnecessary for QA.
# Show the UAV footprint on the EE map and use the two local plots above for exact target alignment.
m.addLayer(roi, {"color": "yellow"}, "UAV / embedding ROI")
display(m)

print("QA note: satellite imagery is a cloud-filtered median near the UAV survey dates,")
print("while Google Satellite Embedding V1 is annual (2020), not a single-month image.")

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 06 — TARGET DISTRIBUTION")
print("=" * 100)

y_all = obs[good]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_all, bins=50)
axes[0].set_xlabel("Mean UAV Musanga probability at 10 m")
axes[0].set_ylabel("Number of cells")
axes[0].set_title("Full distribution")

positive_view = y_all[y_all > 0.001]
if len(positive_view):
    axes[1].hist(positive_view, bins=50)
axes[1].set_xlabel("Mean UAV Musanga probability")
axes[1].set_title("Cells > 0.001 (display only)")
plt.tight_layout()
plt.show()

for q in [0, .25, .5, .75, .9, .95, .99, 1]:
    print(f"q{q:>4}: {np.quantile(y_all, q):.6f}")

## BLOCK 07 — Build canonical 10-m training table

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 07 — BUILD TRAINING TABLE")
print("=" * 100)

with rasterio.open(EMBED_TIF) as src:
    Xcube = src.read().astype("float32")
    emb_nodata = src.nodata
    transform = src.transform

emb_valid = np.all(np.isfinite(Xcube), axis=0)
if emb_nodata is not None:
    emb_valid &= np.all(Xcube != emb_nodata, axis=0)

mask = good & emb_valid
rows, cols = np.where(mask)
X = Xcube[:, mask].T
target = obs[mask]
xs, ys = rasterio.transform.xy(transform, rows, cols, offset="center")
df = pd.DataFrame({
    "cell_id": np.arange(len(target), dtype=int),
    "x": np.asarray(xs),
    "y": np.asarray(ys),
    "musanga_prob_mean": target,
})
for i in range(64):
    df[f"A{i:02d}"] = X[:, i]

df.to_csv(TABLE_CSV, index=False)
print("Rows:", len(df))
print("Only cells with a finite UAV target AND all 64 finite embedding bands are included.")
print("NoData cells are excluded, not assigned target=0.")
print("Predictors:", X.shape[1])
print("Saved:", TABLE_CSV)
display(df.head())

## BLOCK 08 — Random Yoko train/test benchmark

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 08 — RANDOM TRAIN/TEST BENCHMARK")
print("=" * 100)

idx = np.arange(len(target))
train_idx, test_idx = train_test_split(
    idx, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=400, min_samples_leaf=2, max_features="sqrt",
        n_jobs=-1, random_state=RANDOM_SEED
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=250, learning_rate=0.05, l2_regularization=1.0,
        random_state=RANDOM_SEED
    ),
}

results, fitted = [], {}
for name, model in models.items():
    model.fit(X[train_idx], target[train_idx])
    pred = np.clip(model.predict(X[test_idx]), 0, 1)
    results.append({
        "model": name,
        "RMSE": mean_squared_error(target[test_idx], pred) ** 0.5,
        "MAE": mean_absolute_error(target[test_idx], pred),
        "R2": r2_score(target[test_idx], pred),
        "Spearman_rho": spearmanr(target[test_idx], pred).statistic,
    })
    fitted[name] = model

baseline = np.full(len(test_idx), target[train_idx].mean())
results.append({
    "model": "Mean_baseline",
    "RMSE": mean_squared_error(target[test_idx], baseline) ** 0.5,
    "MAE": mean_absolute_error(target[test_idx], baseline),
    "R2": r2_score(target[test_idx], baseline),
    "Spearman_rho": np.nan,
})

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)

candidate_results = results_df[results_df["model"] != "Mean_baseline"]
best_name = candidate_results.iloc[0]["model"]
best_model = fitted[best_name]
print("Selected model:", best_name)

# Keep the held-out predictions for the diagnostic block.
test_observed = target[test_idx].copy()
test_predictions = {
    name: np.clip(model.predict(X[test_idx]), 0, 1)
    for name, model in fitted.items()
}
best_test_pred = test_predictions[best_name]


## BLOCK 08B — Abundance-stratified held-out diagnostic\n\nEvaluate M01 separately across observed UAV Musanga abundance classes. Bins are descriptive only and do not alter model training or the continuous target.\n

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 08B — ABUNDANCE-STRATIFIED HELD-OUT DIAGNOSTIC")
print("=" * 100)

y_true = test_observed
y_pred = best_test_pred

# Ecologically interpretable descriptive bins; model remains continuous.
edges = np.array([0.0, 0.01, 0.05, 0.20, 0.50, 1.000001])
labels = ["0–0.01", "0.01–0.05", "0.05–0.20", "0.20–0.50", ">0.50"]

bin_id = np.digitize(y_true, edges[1:-1], right=False)
diag_rows = []
for b, label in enumerate(labels):
    m = bin_id == b
    n = int(m.sum())
    if n == 0:
        continue
    yt, yp = y_true[m], y_pred[m]
    diag_rows.append({
        "observed_bin": label,
        "n": n,
        "fraction_test": n / len(y_true),
        "observed_mean": float(np.mean(yt)),
        "predicted_mean": float(np.mean(yp)),
        "bias_pred_minus_obs": float(np.mean(yp - yt)),
        "MAE": float(mean_absolute_error(yt, yp)),
        "RMSE": float(mean_squared_error(yt, yp) ** 0.5),
        "Spearman_rho": float(spearmanr(yt, yp).statistic) if n >= 3 else np.nan,
    })

abundance_diag = pd.DataFrame(diag_rows)
display(abundance_diag)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_true, y_pred, s=10, alpha=0.35)
ax.plot([0, 1], [0, 1], "--", linewidth=1)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Observed mean UAV Musanga probability")
ax.set_ylabel("Predicted probability")
ax.set_title(f"M01 held-out predictions — {best_name}")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
xpos = np.arange(len(abundance_diag))
ax.plot(xpos, abundance_diag["observed_mean"], marker="o", label="Observed mean")
ax.plot(xpos, abundance_diag["predicted_mean"], marker="o", label="Predicted mean")
ax.set_xticks(xpos)
ax.set_xticklabels(abundance_diag["observed_bin"], rotation=30, ha="right")
ax.set_ylabel("Mean Musanga probability")
ax.set_xlabel("Observed abundance class")
ax.set_title("M01 response across Musanga abundance")
ax.legend()
plt.tight_layout()
plt.show()

print("\nInterpretation aid:")
print("- predicted_mean << observed_mean in high bins = underestimation of Musanga-rich cells")
print("- predicted_mean >> observed_mean in low bins = false-positive / smoothing tendency")
print("- Spearman shows within-bin ranking skill; n should be considered for rare high bins")


## BLOCK 08C — M02 abundance-weighted embedding regression

M02 keeps the **same 64 Google Satellite Embedding predictors and the identical held-out test cells as M01**, but weights the training loss so rare Musanga-rich cells contribute more strongly. The continuous target is unchanged and the test set is never weighted.

Weights are inverse-frequency by the same five abundance bins used in Block 08B, normalized to mean 1 and capped to avoid letting the rarest bin dominate. Random Forest is used because it was the best M01 architecture.

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 08C — M02 ABUNDANCE-WEIGHTED EMBEDDING REGRESSION")
print("=" * 100)

# Same abundance classes as Block 08B. These bins define TRAINING WEIGHTS only;
# the regression target itself remains continuous.
weight_edges = np.array([0.0, 0.01, 0.05, 0.20, 0.50, 1.000001])
weight_labels = ["0–0.01", "0.01–0.05", "0.05–0.20", "0.20–0.50", ">0.50"]
y_train = target[train_idx]
train_bin = np.digitize(y_train, weight_edges[1:-1], right=False)

# Inverse-frequency weighting gives each abundance class approximately equal
# total influence before capping. Normalize to mean 1 for interpretability.
counts = np.bincount(train_bin, minlength=len(weight_labels))
raw_bin_weights = len(y_train) / (len(weight_labels) * np.maximum(counts, 1))
sample_weight = raw_bin_weights[train_bin]
sample_weight /= sample_weight.mean()

# Cap extreme weights, then renormalize. This is intentionally conservative.
WEIGHT_CAP = 10.0
sample_weight = np.minimum(sample_weight, WEIGHT_CAP)
sample_weight /= sample_weight.mean()

weight_table = pd.DataFrame({
    "observed_bin": weight_labels,
    "n_train": counts,
    "fraction_train": counts / len(y_train),
    "raw_inverse_frequency_weight": raw_bin_weights,
})
weight_table["effective_mean_weight"] = [
    float(sample_weight[train_bin == b].mean()) if counts[b] else np.nan
    for b in range(len(weight_labels))
]
display(weight_table)
print(f"Sample-weight range: {sample_weight.min():.3f} – {sample_weight.max():.3f}")
print(f"Mean sample weight: {sample_weight.mean():.3f}")

# M02 = same RF architecture as M01, only abundance-weighted fitting differs.
m02_model = RandomForestRegressor(
    n_estimators=400,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
m02_model.fit(X[train_idx], y_train, sample_weight=sample_weight)
m02_pred = np.clip(m02_model.predict(X[test_idx]), 0, 1)

def regression_metrics(y, p):
    return {
        "RMSE": float(mean_squared_error(y, p) ** 0.5),
        "MAE": float(mean_absolute_error(y, p)),
        "R2": float(r2_score(y, p)),
        "Spearman_rho": float(spearmanr(y, p).statistic),
    }

m01_pred = best_test_pred
m01_metrics = regression_metrics(test_observed, m01_pred)
m02_metrics = regression_metrics(test_observed, m02_pred)

m01_m02_overall = pd.DataFrame([
    {"model": f"M01 unweighted {best_name}", **m01_metrics},
    {"model": "M02 abundance-weighted RandomForest", **m02_metrics},
])
print("\nOverall held-out comparison — identical test cells:")
display(m01_m02_overall)

# Identical observed bins for a direct M01 versus M02 comparison.
test_bin = np.digitize(test_observed, weight_edges[1:-1], right=False)
comparison_rows = []
for b, label in enumerate(weight_labels):
    m = test_bin == b
    n = int(m.sum())
    if n == 0:
        continue
    yt = test_observed[m]
    p1 = m01_pred[m]
    p2 = m02_pred[m]
    comparison_rows.append({
        "observed_bin": label,
        "n_test": n,
        "observed_mean": float(yt.mean()),
        "M01_pred_mean": float(p1.mean()),
        "M02_pred_mean": float(p2.mean()),
        "M01_bias": float(np.mean(p1 - yt)),
        "M02_bias": float(np.mean(p2 - yt)),
        "M01_MAE": float(mean_absolute_error(yt, p1)),
        "M02_MAE": float(mean_absolute_error(yt, p2)),
        "M01_RMSE": float(mean_squared_error(yt, p1) ** 0.5),
        "M02_RMSE": float(mean_squared_error(yt, p2) ** 0.5),
        "M01_Spearman": float(spearmanr(yt, p1).statistic) if n >= 3 else np.nan,
        "M02_Spearman": float(spearmanr(yt, p2).statistic) if n >= 3 else np.nan,
    })

m01_m02_by_abundance = pd.DataFrame(comparison_rows)
print("\nAbundance-stratified held-out comparison:")
display(m01_m02_by_abundance)

fig, ax = plt.subplots(figsize=(8, 4))
xpos = np.arange(len(m01_m02_by_abundance))
ax.plot(xpos, m01_m02_by_abundance["observed_mean"], marker="o", label="Observed")
ax.plot(xpos, m01_m02_by_abundance["M01_pred_mean"], marker="o", label="M01 unweighted")
ax.plot(xpos, m01_m02_by_abundance["M02_pred_mean"], marker="o", label="M02 weighted")
ax.set_xticks(xpos)
ax.set_xticklabels(m01_m02_by_abundance["observed_bin"], rotation=30, ha="right")
ax.set_xlabel("Observed Musanga abundance class")
ax.set_ylabel("Mean Musanga probability")
ax.set_title("M01 vs M02 — identical held-out cells")
ax.legend()
plt.tight_layout()
plt.show()

print("\nM02 interpretation:")
print("- Improvement in the high-abundance bins indicates that class imbalance was suppressing the M01 upper tail.")
print("- A large increase in the 0–0.01 predicted mean indicates the weighting is buying sensitivity at the cost of false-positive inflation.")
print("- Overall RMSE may worsen even when the Musanga-rich tail improves; judge both the overall and stratified results.")


## BLOCK 08D — M03 moderate abundance-weighted embedding regression

M03 keeps the **same Random Forest, predictors, train/test split, and continuous target** as M01/M02, but uses the square root of the inverse-frequency abundance weights. This is a deliberately milder weighting experiment: it tests whether some upper-tail recovery can be retained without the low-abundance inflation seen in M02.


In [ ]:
print("\n" + "=" * 100)
print("BLOCK 08D — M03 MODERATE ABUNDANCE-WEIGHTED EMBEDDING REGRESSION")
print("=" * 100)

# Reuse the exact M02 training bins/counts and identical M01/M02 held-out cells.
# sqrt(inverse-frequency) moderates the ~40x M02 weight range.
m03_bin_weights = np.sqrt(raw_bin_weights)
m03_sample_weight = m03_bin_weights[train_bin]
m03_sample_weight /= m03_sample_weight.mean()

m03_weight_table = pd.DataFrame({
    "observed_bin": weight_labels,
    "n_train": counts,
    "fraction_train": counts / len(y_train),
    "sqrt_inverse_frequency_weight_raw": m03_bin_weights,
})
m03_weight_table["effective_mean_weight"] = [
    float(m03_sample_weight[train_bin == b].mean()) if counts[b] else np.nan
    for b in range(len(weight_labels))
]
display(m03_weight_table)
print(f"Sample-weight range: {m03_sample_weight.min():.3f} – {m03_sample_weight.max():.3f}")
print(f"Mean sample weight: {m03_sample_weight.mean():.3f}")

m03_model = RandomForestRegressor(
    n_estimators=400,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
m03_model.fit(X[train_idx], y_train, sample_weight=m03_sample_weight)
m03_pred = np.clip(m03_model.predict(X[test_idx]), 0, 1)
m03_metrics = regression_metrics(test_observed, m03_pred)

m01_m02_m03_overall = pd.DataFrame([
    {"model": f"M01 unweighted {best_name}", **m01_metrics},
    {"model": "M02 inverse-frequency weighted RF", **m02_metrics},
    {"model": "M03 sqrt-inverse-frequency weighted RF", **m03_metrics},
])
print("\nOverall held-out comparison — identical test cells:")
display(m01_m02_m03_overall)

comparison_rows = []
for b, label in enumerate(weight_labels):
    m = test_bin == b
    n = int(m.sum())
    if n == 0:
        continue
    yt = test_observed[m]
    p1, p2, p3 = m01_pred[m], m02_pred[m], m03_pred[m]
    row = {
        "observed_bin": label,
        "n_test": n,
        "observed_mean": float(yt.mean()),
    }
    for name, p in [("M01", p1), ("M02", p2), ("M03", p3)]:
        row[f"{name}_pred_mean"] = float(p.mean())
        row[f"{name}_bias"] = float(np.mean(p - yt))
        row[f"{name}_MAE"] = float(mean_absolute_error(yt, p))
        row[f"{name}_RMSE"] = float(mean_squared_error(yt, p) ** 0.5)
        row[f"{name}_Spearman"] = float(spearmanr(yt, p).statistic) if n >= 3 else np.nan
    comparison_rows.append(row)

m01_m02_m03_by_abundance = pd.DataFrame(comparison_rows)
print("\nAbundance-stratified held-out comparison:")
display(m01_m02_m03_by_abundance)

fig, ax = plt.subplots(figsize=(8, 4))
xpos = np.arange(len(m01_m02_m03_by_abundance))
ax.plot(xpos, m01_m02_m03_by_abundance["observed_mean"], marker="o", label="Observed")
ax.plot(xpos, m01_m02_m03_by_abundance["M01_pred_mean"], marker="o", label="M01 unweighted")
ax.plot(xpos, m01_m02_m03_by_abundance["M02_pred_mean"], marker="o", label="M02 inverse-weighted")
ax.plot(xpos, m01_m02_m03_by_abundance["M03_pred_mean"], marker="o", label="M03 sqrt-weighted")
ax.set_xticks(xpos)
ax.set_xticklabels(m01_m02_m03_by_abundance["observed_bin"], rotation=30, ha="right")
ax.set_xlabel("Observed Musanga abundance class")
ax.set_ylabel("Mean Musanga probability")
ax.set_title("M01 vs M02 vs M03 — identical held-out cells")
ax.legend()
plt.tight_layout()
plt.show()

print("\nM03 interpretation:")
print("- M03 is successful if it improves moderate/high-abundance predictions relative to M01")
print("  while producing substantially less low-abundance inflation than M02.")
print("- If the trade-off remains weak, stop tuning embedding weights and move to Sentinel-2 temporal/spectral predictors.")


## BLOCK 09 — Held-out diagnostic plots

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 09 — HELD-OUT DIAGNOSTICS")
print("=" * 100)

test_pred = np.clip(best_model.predict(X[test_idx]), 0, 1)
test_obs = target[test_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(test_obs, test_pred, s=8, alpha=0.35)
lim = max(float(test_obs.max()), float(test_pred.max()))
axes[0].plot([0, lim], [0, lim], "--")
axes[0].set_xlabel("Observed UAV mean probability")
axes[0].set_ylabel("Predicted probability")
axes[0].set_title(f"Held-out cells — {best_name}")

res = test_obs - test_pred
axes[1].scatter(test_pred, res, s=8, alpha=0.35)
axes[1].axhline(0, linestyle="--")
axes[1].set_xlabel("Predicted probability")
axes[1].set_ylabel("Observed − predicted")
axes[1].set_title("Residuals")
plt.tight_layout()
plt.show()

## BLOCK 10 — Refit on all Yoko cells and export maps

In [ ]:
print("\n" + "=" * 100)
print("BLOCK 10 — REFIT / EXPORT MAPS")
print("=" * 100)

final_model = models[best_name]
final_model.fit(X, target)
joblib.dump(final_model, MODEL_FILE)

pred_all = np.clip(final_model.predict(X), 0, 1).astype("float32")
pred_grid = np.full(dst_shape, -9999.0, dtype="float32")
resid_grid = np.full(dst_shape, -9999.0, dtype="float32")
pred_grid[rows, cols] = pred_all
resid_grid[rows, cols] = target - pred_all

for path, arr in [(PRED_TIF, pred_grid), (RESID_TIF, resid_grid)]:
    with rasterio.open(path, "w", **profile1) as dst:
        dst.write(arr, 1)

summary = {
    "site": "Yoko",
    "embedding_year": EMBEDDING_YEAR,
    "target": "mean UAV-derived Musanga probability on Google embedding grid",
    "threshold_used_for_target": False,
    "n_cells": int(len(target)),
    "test_fraction": TEST_SIZE,
    "split": "random single-site development split",
    "selected_model": best_name,
    "benchmark": results_df.replace({np.nan: None}).to_dict(orient="records"),
    "outputs": {
        "observed_10m": str(OBS_TIF),
        "predicted_10m": str(PRED_TIF),
        "residual_10m": str(RESID_TIF),
        "training_table": str(TABLE_CSV),
        "model": str(MODEL_FILE),
    },
}
with open(SUMMARY_JSON, "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nSaved prediction:", PRED_TIF)
print("Saved residual:", RESID_TIF)
print("Saved summary:", SUMMARY_JSON)

## BLOCK 11 — Interpretation and next step

This notebook is a **Yoko development test**, not yet a test of geographic transferability.

The random held-out metrics test whether the 2020 Google Satellite Embedding contains enough local information to reconstruct variation in the UAV-derived Musanga signal at Yoko. Nearby pixels are spatially related, so these metrics should not be interpreted as independent regional validation.

The stronger next experiment is **cross-site transfer**: train at Yoko and predict an independent site with its own UAV Musanga probability raster.

If the embedding-only experiment is promising, the next benchmark should compare:
- Google Satellite Embedding only;
- explicit Sentinel-2 temporal/spectral features;
- Satellite Embedding + Sentinel-2 temporal features.